### Transformação de dados na Camada Silver

Leitura dos dados da camada Bronze, aplicação das regras de qualidade e integridade referencial, e armazenamento dos dados tratados na camada Silver.

In [0]:
# CARREGAMENTO DE VARIÁVEIS DE AMBIENTE
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, current_timestamp, year, month, trim, when, lit, row_number, to_date, to_timestamp,coalesce
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType, TimestampType
from delta.tables import DeltaTable

load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

# DICIONÁRIO DE CREDENCIAIS OAUTH PARA USAR COM SPARK
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# DICIONÁRIO DE PATHS
paths = {
    "food_avaliacoes_produto_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_avaliacoes_produto",
    "food_estoque_lojas_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_estoque_lojas",
    "food_avaliacoes_produto_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_avaliacoes_produto",
    "food_estoque_lojas_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_estoque_lojas",
    "raw_path": f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data"
}

In [0]:
# LEITURA DAS TABELAS BRONZE
df_bronze_avaliacoes = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_avaliacoes_produto_bronze"])
)

df_bronze_estoque = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_estoque_lojas_bronze"])
)

df_bronze_avaliacoes.printSchema()
df_bronze_estoque.printSchema()
display(df_bronze_avaliacoes.limit(5))
display(df_bronze_estoque.limit(5))
df_bronze_avaliacoes.groupBy("nota").count().orderBy("nota").show()

In [0]:
# LEITURA DAS TABELAS DE REFERÊNCIA DA RAW

# SQUAD 3
df_physical_lojas = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_lojas.csv")
)

df_physical_produtos_pereciveis = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_produtos_pereciveis.csv")
)

df_food_lotes_producao = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/food_lotes_producao.csv")
)

# SQUAD 1
df_ecommerce_produtos = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_produtos.csv")
)

df_ecommerce_pedidos = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_pedidos.csv")
)

df_physical_lojas.printSchema()
df_ecommerce_produtos.printSchema()
df_physical_produtos_pereciveis.printSchema()
df_food_lotes_producao.printSchema()
df_ecommerce_pedidos.printSchema()


In [0]:
# CHAVES DE REFERÊNCIA

# Lojas Válidas
df_lojas_validas = (
    df_physical_lojas
    .select(col("id_loja"))
    .filter(col("id_loja").isNotNull())
    .dropDuplicates(["id_loja"])
)

# SKUS do Ecommerce
df_skus_ecommerce = (
    df_ecommerce_produtos
    .select(trim(col("sku")).alias("sku"))
    .filter(col("sku").isNotNull())
    .filter(col("sku") != "")
    .dropDuplicates(["sku"])
)
# SKUS dos Produtos Perecíveis
df_skus_pereciveis = (
    df_physical_produtos_pereciveis
    .select(trim(col("sku")).alias("sku"))
    .filter(col("sku").isNotNull())
    .filter(col("sku") != "")
    .dropDuplicates(["sku"])
)
# Catálago Unificado
df_catalogo_unificado = (
    df_skus_ecommerce
    .unionByName(df_skus_pereciveis)
    .dropDuplicates(["sku"])
)
print(f"SKUs ecommerce: {df_skus_ecommerce.count()}")
print(f"SKUs perecíveis: {df_skus_pereciveis.count()}")
print(f"SKUs catálogo unificado: {df_catalogo_unificado.count()}")

# PEDIDOS VÁLIDOS
df_pedidos_validos = (
    df_ecommerce_pedidos
    .select(col("id_pedido"))
    .filter(col("id_pedido").isNotNull())
    .dropDuplicates(["id_pedido"])
)

# LOTES VÁLIDOS
df_lotes_validos = (
    df_food_lotes_producao
    .select(
        col("id_lote"),
        trim(col("sku")).alias("sku"),
        to_timestamp(col("dt_fabricacao")).alias("dt_fabricacao"),
        to_timestamp(col("dt_validade")).alias("dt_validade")
    )
    .filter(col("id_lote").isNotNull())
    .dropDuplicates(["id_lote"])
)

# ESTOQUE COM LOTES VÁLIDOS
df_estoque_lotes = (
    df_bronze_estoque.alias("e")
    .join(
        df_lotes_validos.alias("l"),
        (col("e.id_lote") == col("l.id_lote")) &
        (to_timestamp(col("e.dt_snapshot")) >= col("l.dt_fabricacao")) &
        (to_timestamp(col("e.dt_snapshot")) <= col("l.dt_validade")),
        "inner"
    )
)

print(f"Registros de estoque com lote e snapshot válidos: {df_estoque_lotes.count()}")


In [0]:
# SILVER - FOOD ESTOQUE LOJAS
df_silver_estoque = (
    df_bronze_estoque
    .withColumn("sku",trim(col("sku")))
    .withColumn("id_lote",col("id_lote").cast("integer"))
)

# REGRA 1
# id_loja deve existir em physical_lojas
df_silver_estoque = (
    df_silver_estoque.alias("e")
    .join(df_lojas_validas.alias("l"),col("e.id_loja") == col("l.id_loja"),"left")
    .withColumn("silver_flag_id_loja_invalido",when(col("e.id_loja").isNull() |col("l.id_loja").isNull(),lit(True)).otherwise(lit(False)))
    .drop(col("l.id_loja")
    )
)

# REGRA 2
# sku deve existir no catálogo unificado
df_silver_estoque = (
    df_silver_estoque.alias("e")
    .join(df_catalogo_unificado.alias("c"),col("e.sku") == col("c.sku"),"left")
    .withColumn("silver_flag_sku_invalido",when(col("e.sku").isNull() |col("c.sku").isNull(),lit(True)).otherwise(lit(False)))
    .drop(col("c.sku"))
)

# REGRA 3
# quantidade_disponivel deve ser >= 0
df_silver_estoque = df_silver_estoque.withColumn(
        "silver_flag_quantidade_negativa",
        when(col("quantidade_disponivel").isNull() |(col("quantidade_disponivel") < 0),lit(True)).otherwise(lit(False))
)

# REGRAS 4 E 5
df_silver_estoque = (
    df_silver_estoque.alias("e")
    .join(
        df_lotes_validos.alias("l"),
        col("e.id_lote") == col("l.id_lote"),
        "left"
    )

    # REGRA 4
    # Quando id_lote é preenchido, deve existir em food_lotes_producao
    .withColumn(
        "silver_flag_id_lote_invalido",
        when(col("e.id_lote").isNotNull() & col("l.id_lote").isNull(),lit(True)).otherwise(lit(False))
    )

    # REGRA 5
    # dt_snapshot deve estar dentro do intervalo [dt_fabricacao, dt_validade] do lote referenciado
    .withColumn(
        "silver_flag_dt_snapshot_invalido",
        when(
            col("e.id_lote").isNotNull() &
            col("l.id_lote").isNotNull() &
            (
                col("l.dt_fabricacao").isNull() |
                col("l.dt_validade").isNull() |
                (
                    to_timestamp(col("e.dt_snapshot"))
                    < col("l.dt_fabricacao")
                ) |
                (
                    to_timestamp(col("e.dt_snapshot"))
                    > col("l.dt_validade")
                )
            ),
            lit(True)
        ).otherwise(
            lit(False)
        )
    )
    # REMOÇÃO DAS COLUNAS AUXILIARES
    .drop(col("l.id_lote"),col("l.sku"),col("l.dt_fabricacao"),col("l.dt_validade"))
)


# STATUS GERAL DE QUALIDADE
# FOOD ESTOQUE LOJAS
df_silver_estoque = (
    df_silver_estoque
    .withColumn(
        "silver_quality_issue",
        (
            col("silver_flag_id_loja_invalido") |
            col("silver_flag_sku_invalido") |
            col("silver_flag_quantidade_negativa") |
            col("silver_flag_id_lote_invalido") |
            col("silver_flag_dt_snapshot_invalido")
        )
    )
    .withColumn(
        "silver_quality_status",
        when(
            col("silver_quality_issue"),
            lit("REVIEW")
        ).otherwise(
            lit("OK")
        )
    )
)



In [0]:
# SILVER - FOOD AVALIACOES PRODUTO
df_silver_avaliacoes = (
    df_bronze_avaliacoes
    .withColumn("sku",trim(col("sku")))
    .withColumn("id_pedido",col("id_pedido").cast("integer"))
)


# REGRA 1
# REMOÇÃO DE id_avaliacao NULO
total_antes = df_silver_avaliacoes.count()

df_silver_avaliacoes = (
    df_silver_avaliacoes.filter(col("id_avaliacao").isNotNull())
)

total_depois_nulos = df_silver_avaliacoes.count()
print(f"Total antes: {total_antes}")
print(f"Total após remoção de id_avaliacao nulo: {total_depois_nulos}")
print(f"IDs nulos removidos: {total_antes - total_depois_nulos}")

# REMOÇÃO DE DUPLICIDADES
window_pk = (
    Window
    .partitionBy("id_avaliacao")
    .orderBy(col("bronze_ingested_at").desc(),col("bronze_source_file").desc()))

df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn("_rn",row_number().over(window_pk))
    .filter(col("_rn") == 1)
    .drop("_rn")
)


# REGRA 2
# nota deve ser um inteiro entre 1 e 5
df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "silver_flag_nota_invalida",
        when(col("nota").isNull() |(col("nota") < 1) |(col("nota") > 5),lit(True)).otherwise(lit(False)))
)


# REGRA 3
# verificada = TRUE
df_silver_avaliacoes = (
    df_silver_avaliacoes.alias("a")
    .join(df_pedidos_validos.alias("p"), col("a.id_pedido") == col("p.id_pedido"), "left")
    .withColumn(
        "silver_flag_pedido_verificado_invalido",
        when(
            (col("a.verificada") == True) & (col("a.id_pedido").isNull() | col("p.id_pedido").isNull()),
            lit(True)
        ).otherwise(lit(False))
    )
    .drop(col("p.id_pedido"))
)


# REGRA 4
# verificada = FALSE
df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "silver_flag_pedido_nao_verificado_invalido",
        when(
            col("verificada") == False,
            when(col("id_pedido").isNotNull(),lit(True)).otherwise(lit(False))
        ).otherwise(lit(False))
    )
)


# REGRA 5
# sku deve existir no catálogo unificado
df_silver_avaliacoes = (
    df_silver_avaliacoes.alias("a")
    .join(
        df_catalogo_unificado.alias("c"),
        col("a.sku") == col("c.sku"),
        "left"
    )
    .withColumn(
        "silver_flag_sku_invalido",
        when(
            col("a.sku").isNull() |
            col("c.sku").isNull(),
            lit(True)
        ).otherwise(lit(False))
    )
    .drop(col("c.sku"))
)


# STATUS GERAL DE QUALIDADE
# FOOD AVALIACOES PRODUTO
df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "silver_quality_issue",
        (
            col("silver_flag_nota_invalida") |
            col("silver_flag_pedido_verificado_invalido") |
            col("silver_flag_pedido_nao_verificado_invalido") |
            col("silver_flag_sku_invalido")
        )
    )
    .withColumn(
        "silver_quality_status",
        when(col("silver_quality_issue"),lit("REVIEW")).otherwise(lit("OK"))
    )
)



In [0]:
# AUDITORIA DA SILVER
df_silver_estoque = (
    df_silver_estoque
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .withColumn(
        "ano",
        year(col("silver_processed_at"))
    )
    .withColumn(
        "mes",
        month(col("silver_processed_at"))
    )
)

df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .withColumn(
        "ano",
        year(col("silver_processed_at"))
    )
    .withColumn(
        "mes",
        month(col("silver_processed_at"))
    )
)

In [0]:
# VALIDAÇÃO - FOOD ESTOQUE LOJAS
df_silver_estoque.groupBy("silver_quality_status").count().show()

df_silver_estoque.groupBy(
    "silver_flag_id_loja_invalido",
    "silver_flag_sku_invalido",
    "silver_flag_quantidade_negativa",
    "silver_flag_id_lote_invalido",
    "silver_flag_dt_snapshot_invalido"
).count().show()

# display(df_silver_estoque.filter(col("silver_quality_issue") == True))
display(df_silver_estoque)


# VALIDAÇÃO - FOOD AVALIACOES PRODUTO
df_silver_avaliacoes.groupBy("silver_quality_status").count().show()

df_silver_avaliacoes.groupBy(
    "silver_flag_nota_invalida",
    "silver_flag_pedido_verificado_invalido",
    "silver_flag_pedido_nao_verificado_invalido",
    "silver_flag_sku_invalido"
).count().show()

# display(df_silver_avaliacoes.filter(col("silver_quality_issue") == True))
display(df_silver_avaliacoes)

In [0]:
display(
    df_silver_estoque
    .select(
        when(col("id_lote").isNull(), lit("id_lote NULL"))
        .when(
            col("silver_flag_id_lote_invalido") == False,
            lit("Preenchido e existe em food_lotes_producao")
        )
        .otherwise(
            lit("Preenchido e não existe em food_lotes_producao")
        )
        .alias("situacao_id_lote")
    )
    .groupBy("situacao_id_lote")
    .count()
    .orderBy("situacao_id_lote")
)

In [0]:
# GRAVAÇÃO SILVER - FOOD AVALIAÇÕES PRODUTO
# Upsert lógico via LEFT ANTI + UNION
try:
    df_avaliacoes_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["food_avaliacoes_produto_silver"])
    )

    # Confirma que a leitura realmente conseguiu acessar a Silver
    df_avaliacoes_atual.limit(1).collect()

    silver_avaliacoes_existe = True

except Exception:
    silver_avaliacoes_existe = False


if not silver_avaliacoes_existe:

    # Primeira carga
    df_avaliacoes_final = df_silver_avaliacoes

    print("Silver de food_avaliacoes_produto não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    # Chaves que chegaram nesta carga
    chaves_avaliacoes = (
        df_silver_avaliacoes
        .select("id_avaliacao")
        .dropDuplicates()
    )

    # Mantém somente registros antigos que NÃO chegaram novamente
    df_avaliacoes_mantidas = (
        df_avaliacoes_atual
        .join(
            chaves_avaliacoes,
            on="id_avaliacao",
            how="left_anti"
        )
    )

    # Junta:
    # - registros novos/atualizados
    # - registros antigos que não foram afetados
    df_avaliacoes_final = (
        df_silver_avaliacoes
        .unionByName(df_avaliacoes_mantidas)
    )

    print("Silver de food_avaliacoes_produto já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


# Grava a Silver
(
    df_avaliacoes_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .partitionBy("ano", "mes")
    .save(paths["food_avaliacoes_produto_silver"])
)

print("✅ Silver de food_avaliacoes_produto gravada com sucesso.")


# GRAVAÇÃO SILVER - FOOD ESTOQUE LOJAS
# Upsert lógico via LEFT ANTI + UNION
try:
    df_estoque_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["food_estoque_lojas_silver"])
    )

    # Confirma que a leitura realmente conseguiu acessar a Silver
    df_estoque_atual.limit(1).collect()

    silver_estoque_existe = True

except Exception:
    silver_estoque_existe = False


if not silver_estoque_existe:

    # Primeira carga
    df_estoque_final = df_silver_estoque

    print("Silver de food_estoque_lojas não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    # Chaves que chegaram nesta carga
    chaves_estoque = (
        df_silver_estoque
        .select(
            "id_loja",
            "sku",
            "dt_snapshot"
        )
        .dropDuplicates()
    )

    # Mantém somente registros antigos que NÃO chegaram novamente
    df_estoque_mantidas = (
        df_estoque_atual
        .join(
            chaves_estoque,
            on=[
                "id_loja",
                "sku",
                "dt_snapshot"
            ],
            how="left_anti"
        )
    )

    # Junta:
    # - registros novos/atualizados
    # - registros antigos que não foram afetados
    df_estoque_final = (
        df_silver_estoque
        .unionByName(df_estoque_mantidas)
    )

    print("Silver de food_estoque_lojas já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


# Grava a Silver
(
    df_estoque_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .partitionBy("ano", "mes")
    .save(paths["food_estoque_lojas_silver"])
)

print("✅ Silver de food_estoque_lojas gravada com sucesso.")

In [0]:
# LEITURA FINAL DA SILVER
df_final_avaliacoes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(paths["food_avaliacoes_produto_silver"])
)

df_final_estoque = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(paths["food_estoque_lojas_silver"])
)


# VALIDAÇÃO FINAL
print(f"food_avaliacoes_produto Silver: {df_final_avaliacoes.count()} registros")
print(f"food_estoque_lojas Silver: {df_final_estoque.count()} registros")


# DISTRIBUIÇÃO DA QUALIDADE
print("FOOD AVALIACOES PRODUTO")
df_final_avaliacoes.groupBy("silver_quality_status").count().show()
print("FOOD ESTOQUE LOJAS")
df_final_estoque.groupBy("silver_quality_status").count().show()
      